<a href="https://colab.research.google.com/github/warstlcm/ISA444FinalProject/blob/main/ISA444_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Option 1: Hotel Demand Forecasting

Target (y): Daily room demand

Unique Identifier (unique_id): The id for each of the 19 hotels

Horizon (h): 28 days

In [ ]:
# Installs
!pip install statsforecast mlforecast neuralforecast utilsforecast timecopilot lightgbm -U

# Imports
from utilsforecast.preprocessing import fill_gaps
import pandas as pd
import numpy as np
from functools import partial
import matplotlib.pyplot as plt

from statsforecast import StatsForecast
from mlforecast import MLForecast
from neuralforecast import NeuralForecast
from timecopilot import TimeCopilotForecaster

from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoARIMA, RandomWalkWithDrift
from lightgbm import LGBMRegressor
from neuralforecast.auto import AutoNBEATS, AutoNHITS
from timecopilot.models.foundation.chronos import Chronos

from utilsforecast.plotting import plot_series
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse, mape, bias, mase

### Preparation and validation of the data

In [ ]:
df = pd.read_parquet('sample_hotels.parquet')
df['ds'] = pd.to_datetime(df['ds'])
df['y'] = pd.to_numeric(df['y'])

static_feats = ['location_type', 'hotel_type']
otb_cols = [f'otb_{i}' for i in range(1, 61)]
exog_feats = ['holiday_flag'] + otb_cols

df_simple = df[['unique_id', 'ds', 'y']]

display(df_simple)

### Preliminary Visualization

In [ ]:
plot_series(df_simple, max_ids=19, plot_random=False)


### 5-Fold Time-Series Cross-Validation

In [ ]:
# 1. Baselines and StatsForecast
models_sf = [
    Naive(alias='Naive'),
    SeasonalNaive(season_length=7, alias='SeasonalNaive'),
    RandomWalkWithDrift(alias='Drift'),
    AutoETS(season_length=7, alias='AutoETS'),
    AutoARIMA(season_length=7, alias='AutoARIMA')
]
sf = StatsForecast(models=models_sf, freq='D')
cv_sf = sf.cross_validation(df=df_simple, h=28, n_windows=5, step_size=28)

In [ ]:
# 2. MLForecast with LightGBM

df_ml = fill_gaps(
    df,
    id_col='unique_id',
    time_col='ds',
    freq='D'
)

for col in static_feats:
    df_ml[col] = df_ml.groupby('unique_id')[col].ffill().bfill()

df_ml['y'] = df_ml['y'].fillna(0)

if 'holiday_flag' in df_ml.columns and df_ml['holiday_flag'].dtype == 'object':
    df_ml['holiday_flag'] = df_ml['holiday_flag'].map({'yes': 1, 'no': 0}).fillna(-1).astype(int)

for col in ['target_day', 'target_month', 'location_type', 'hotel_type']:
    if col in df_ml.columns and df_ml[col].dtype == 'object':
        df_ml[col] = df_ml[col].astype('category').cat.codes

if 'target_year' in df.columns:
    df_ml['target_year'] = df_ml['ds'].dt.year
    df_ml['target_year'] = df_ml['target_year'].fillna(df_ml['ds'].dt.year)

for col in otb_cols:
    if col in df_ml.columns:
        df_ml[col] = df_ml[col].fillna(0)

mlf = MLForecast(
    models=[LGBMRegressor(random_state=444)],
    freq='D',
    lags=list(range(1, 8))

)
cv_ml = mlf.cross_validation(
    df=df_ml,
    h=28,
    n_windows=5,
    step_size=28,
    static_features=static_feats,
    dropna=True
)

In [ ]:
#3. NeuralForecast
from neuralforecast.auto import AutoNBEATS, AutoNHITS

df_neural = df[['unique_id', 'ds', 'y']].copy()

df_neural['y'] = np.log1p(df_neural['y'].clip(lower=0))

min_len = df_neural.groupby('unique_id').size().min()
print(f"Shortest hotel history: {min_len} days")

models_nf = [
    AutoNBEATS(h=28, num_samples=1),
    AutoNHITS(h=28, num_samples=1)
]
nf = NeuralForecast(models=models_nf, freq='D')

try:
    cv_nf = nf.cross_validation(
        df=df_neural,
        n_windows=5,
        step_size=28
    )
    for col in ['AutoNBEATS', 'AutoNHITS']:
        if col in cv_nf.columns:
            cv_nf[col] = np.expm1(cv_nf[col])
    cv_nf['y'] = np.expm1(cv_nf['y'])

    for col in ['y', 'AutoNBEATS', 'AutoNHITS']:
        if col in cv_nf.columns:
            cv_nf[col] = np.expm1(cv_nf[col])

    print("Success")
except Exception as e:
    print(f"Fail")


In [ ]:
# 4. Foundation Model (TimeCopilot with chronos)
tcf = TimeCopilotForecaster(models=[Chronos(repo_id="amazon/chronos-bolt-small")])
cv_chronos = tcf.cross_validation(df=df_simple, h=28, n_windows=5, step_size=28)

### Aggregate Metrics and Count Wins


In [ ]:
cv_all = cv_sf.merge(cv_ml.drop(columns='y'), on=['unique_id', 'ds', 'cutoff']) \
              .merge(cv_nf.drop(columns='y'), on=['unique_id', 'ds', 'cutoff']) \
              .merge(cv_chronos.drop(columns='y'), on=['unique_id', 'ds', 'cutoff'])

from functools import partial
metrics = [bias, mae, rmse, mape, partial(mase, seasonality=7)]

eval_df = evaluate(df=cv_all, metrics=metrics, train_df=df_simple)

model_cols = eval_df.columns.drop(['unique_id', 'metric', 'cutoff'])
eval_df['best_model'] = eval_df[model_cols].idxmin(axis=1)

win_summary = eval_df.groupby(['metric', 'best_model']).size().unstack(fill_value=0)

print("Winner Model Win Counts")
display(win_summary)

eval_df.to_csv('final_project_metrics.csv', index=False)

In [ ]:
win_summary.plot(kind='bar', figsize=(15, 7), title='Winner Model Counts by Metric and Model', rot=45)
plt.xlabel('Metric')
plt.ylabel('Number of Wins')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**Main Findings:**
LightGBM (MLForecast) was the dominant model, winning 75 of 90 series metric combinations across MAE, MAPE, MASE, and RMSE. This is likely because the OTB which is representing rooms already reserved. This gave it direct access to future demand signals that soley historical models cannot see. The statistical and neural models competed mainly on the bias metric, where AutoNBEATS won 18, suggesting the neural was better in terms of accuracy even when its point forecast magnitude wasn't the lowest.
SeasonalNaive and Naive were competitive on a small number of series with hotels that have very rigid, predictable weekly demand patterns.

**MAPE:**
Several hotels have demand values of 0 on certain days. MAPE is undefined when the actual value is 0, so the MAPE results should be interpreted with that in mind. Since the demand values are already between 0 and 1 ( 1 means
fully booked), an MAE of 0.05 would mean our forecasts were off by about
5 percent of occupancy on average.

### Final Forecasts and Visualization

In [ ]:
final_sf = sf.forecast(df=df_simple, h=28)

# --- Start MLForecast Prediction Preparation ---

h = 28 # Prediction horizon

# 1. Get the last date for each unique_id from df_ml to define prediction start
last_dates_df_ml = df_ml.groupby('unique_id')['ds'].max().reset_index()

# 2. Create a future dataframe with unique_id and future 'ds' for the horizon
future_rows = []
for _, row in last_dates_df_ml.iterrows():
    uid = row['unique_id']
    last_ds = row['ds']
    future_ds_range = pd.date_range(start=last_ds + pd.Timedelta(days=1), periods=h, freq='D')
    for date_val in future_ds_range:
        future_rows.append({'unique_id': uid, 'ds': date_val})
future_exog_df = pd.DataFrame(future_rows)

# 3. Populate dynamic exogenous features, replicating preprocessing from df_ml
# holiday_flag: Assume no holidays for future predictions, map to 0 (as 'no' was mapped to 0)
future_exog_df['holiday_flag'] = 0

# target_day, target_month: Replicate categorical encoding using original df values for consistent mapping
day_names_sorted = sorted(df['target_day'].unique())
day_name_to_code_map = {name: i for i, name in enumerate(day_names_sorted)}
future_exog_df['target_day'] = future_exog_df['ds'].dt.day_name().map(day_name_to_code_map).fillna(-1).astype(int)

month_names_sorted = sorted(df['target_month'].unique())
month_name_to_code_map = {name: i for i, name in enumerate(month_names_sorted)}
future_exog_df['target_month'] = future_exog_df['ds'].dt.month_name().map(month_name_to_code_map).fillna(-1).astype(int)

# target_year
future_exog_df['target_year'] = future_exog_df['ds'].dt.year

# otb_cols: Fill with 0 for future predictions, as done during training
for col in otb_cols:
    future_exog_df[col] = 0

# Define the dynamic exogenous columns expected by the model
dynamic_exog_columns = ['holiday_flag', 'target_day', 'target_month', 'target_year'] + otb_cols
# Create the X_df for prediction, containing unique_id, ds, and dynamic exogenous features
X_df_predict = future_exog_df[['unique_id', 'ds'] + dynamic_exog_columns]

# Fit MLForecast before predicting
mlf.fit(df=df_ml, static_features=static_feats)
final_ml = mlf.predict(h=h, X_df=X_df_predict) # Pass X_df for future exogenous features

# --- End MLForecast Prediction Preparation ---

final_nf = nf.predict(df=df_neural)
for col in ['AutoNBEATS', 'AutoNHITS']:
    if col in final_nf.columns:
        final_nf[col] = np.expm1(final_nf[col])

# The TimeCopilotForecaster does not have a 'fit' method; it fits internally during predict.
# Replaced 'predict' with 'forecast' for TimeCopilotForecaster
final_chronos = tcf.forecast(df=df_simple, h=28)

final_all = final_sf \
    .merge(final_ml.drop(columns='y', errors='ignore'), on=['unique_id', 'ds']) \
    .merge(final_nf.drop(columns='y', errors='ignore'), on=['unique_id', 'ds']) \
    .merge(final_chronos.drop(columns='y', errors='ignore'), on=['unique_id', 'ds'])

plot_series(df_simple, final_all, max_ids=19, plot_random=False)

final_all.to_csv('final_test_results.csv', index=False)